# Coffee Plantation Image Classification into Ripe and Unripe

## Comparative Analysis of Machine Learning (Random Forest and SVM) and Deep Learning (CNN) Models

### Step 1: Data Extraction

Dataset Source: This project utilizes the [Drone-based Agricultural Dataset for Crop Yield Estimation provided by KaraAgroAI](https://huggingface.co/datasets/KaraAgroAI/Drone-based-Agricultural-Dataset-for-Crop-Yield-Estimation) 
.
In this step, we see how the images and their labels are arranged inside the file. The file: "Coffee" contains the images in "Ariel" and "side" image folders. 

The Side Image folder has 10 Batches of images, while the Ariel Image format has 5 Bathes of images. Each batch contains images of both "Ripe" and "Unripe" coffee cherries.

There were some Images which were neither Ripe nor Unripe. They were "Spoiled" or "Ripenning". They were so few in number that they could have skewed our models. Hence, we decided to remove them from the dataset.

In [ ]:
# Importing all necessary libraries need for this section

import os # To handle files in the Device
import zipfile # To Unzip ziped files
import numpy as np # For np arrays
from PIL import Image # For Working with images
from collections import Counter

In [ ]:
# Dataset path
dataset_path = '/Applications/Programs for Fun/Code-AI/Project - Image Classification/Coffee'

# Class mapping for YOLO format
class_mapping = {0: "Unripe", 1: "Coffee_tree", 2: "Ripening", 4: "Ripe"}

The Labels are strored in a CSV File in YOLO Format.

The below function loads the images' and their labels in two lists: "image" and "labels". Both the lists have a one-to-one relation.

The parameters passed are the path of the images and the labels inside our local directory

It takes only the images which have labeling "Ripe" and "Unripe"

In [ ]:
def load_image_with_labels(image_path, label_path, class_mapping):
    """
    Load an image and its YOLO format labels
    """
    try:
        # Opening the Image
        image = Image.open(image_path)
        
        # Read labels
        labels = []
        # If file exists
        if os.path.exists(label_path):
            # Open the file
            with open(label_path, 'r') as f:
                # For each line
                for line in f:
                    parts = line.strip().split()
                    if len(parts) == 5:
                        class_id = int(parts[0])
                        class_name = class_mapping.get(class_id, f"Unknown_{class_id}")
                        if class_name in ['Unripe', 'Ripe']:  # Only keep Ripe/Unripe
                            labels.append(class_name)
        
        # Retruns the list of images and thier corresponding labels
        return image, labels
    
    except Exception as e:
        print(f"Error loading {image_path}: {e}")
        return None, []

The below function is defined to load images from the 'side' directory. It calls the load_image_with_label function and the summerizes each of the batches about the total number of images and how many of them are ripe and how many are unripe

In [ ]:
def load_side_images():
    """
    Load all images from side folders with detailed batch statistics
    """
    side_data = []
    side_path = os.path.join(dataset_path, "side")
    
    # Check if side path exists
    if not os.path.exists(side_path):
        print(f"Side path not found: {side_path}")
        return side_data, {}
    
    batch_folders = sorted([item for item in os.listdir(side_path) 
                    if os.path.isdir(os.path.join(side_path, item)) and item.startswith('Batch')])
    
    print(f"Loading {len(batch_folders)} side batches...")
    print("=" * 68)
    
    batch_statistics = {}
    
    # For each batch, we check the image and its label
    for batch in batch_folders:
        images_dir = os.path.join(side_path, batch, "images")
        labels_dir = os.path.join(side_path, batch, "labels")
        
        if not os.path.exists(images_dir):
            print(f"Images directory not found: {images_dir}")
            continue
            
        image_files = [f for f in os.listdir(images_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.JPG'))]
        images_loaded = 0
        batch_class_counts = Counter()
        
        for image_file in image_files:
            image_path = os.path.join(images_dir, image_file)
            label_file = image_file.replace('.JPG', '.txt').replace('.jpg', '.txt').replace('.jpeg', '.txt').replace('.png', '.txt')
            label_path = os.path.join(labels_dir, label_file)
            
            # Try alternative label names
            if not os.path.exists(label_path):
                alt_label_file = "Copy of " + label_file
                alt_label_path = os.path.join(labels_dir, alt_label_file)
                if os.path.exists(alt_label_path):
                    label_path = alt_label_path
                else:
                    continue
            
            image, labels = load_image_with_labels(image_path, label_path, class_mapping)
            
            if image and labels:
                # Use most frequent class as image label
                class_counts = Counter(labels)
                primary_class = max(class_counts.items(), key=lambda x: x[1])[0]
                
                side_data.append({
                    'image_path': image_path,
                    'primary_class': primary_class,
                    'folder': 'side',
                    'batch': batch
                })
                images_loaded += 1
                batch_class_counts[primary_class] += 1
        
        # Store batch statistics
        batch_statistics[batch] = {
            'total': images_loaded,
            'unripe': batch_class_counts['Unripe'],
            'ripe': batch_class_counts['Ripe']
        }
        
        # Print batch details
        total_in_batch = batch_class_counts['Unripe'] + batch_class_counts['Ripe']
        if total_in_batch > 0:
            unripe_pct = (batch_class_counts['Unripe'] / total_in_batch) * 100
            ripe_pct = (batch_class_counts['Ripe'] / total_in_batch) * 100
            print(f"  {batch:8} | {images_loaded:3} images | Unripe: {batch_class_counts['Unripe']:3} ({unripe_pct:5.1f}%) | Ripe: {batch_class_counts['Ripe']:3} ({ripe_pct:5.1f}%)")
        else:
            print(f"  {batch:8} | {images_loaded:3} images | No Ripe/Unripe images")
    
    return side_data, batch_statistics

The below function is defined to load images from the 'Aerial' directory. It calls the load_image_with_label function and the summerizes each of the batches about the total number of images and how many of them are ripe and how many are unripe

In [ ]:
def load_aerial_images():
    """
    Load all images from aerial folders with detailed batch statistics
    """
    aerial_data = []
    aerial_path = os.path.join(dataset_path, "Aerial")
    
    # Check if aerial path exists
    if not os.path.exists(aerial_path):
        print(f"Aerial path not found: {aerial_path}")
        return aerial_data, {}
    
    batches = ['Batch 1', 'Batch 2', 'Batch 3', 'Batch 4', 'Batch 5']
    print(f"\nLoading {len(batches)} aerial batches...")
    print("=" * 68)
    
    batch_statistics = {}
    
    # For each batch,  we check the image and its label
    for batch in batches:
        batch_path = os.path.join(aerial_path, batch)
        
        # Check if batch path exists
        if not os.path.exists(batch_path):
            print(f"  {batch:8} | Batch directory not found")
            continue
        
        # Extract YOLO labels from zip
        yolo_zip_path = os.path.join(batch_path, "labels_yolo.zip")
        if not os.path.exists(yolo_zip_path):
            print(f"  {batch:8} | No labels zip file found")
            continue
            
        # Create temp directory for extracted labels
        temp_labels_dir = os.path.join(batch_path, "extracted_labels")
        os.makedirs(temp_labels_dir, exist_ok=True)
        
        # Extract zip file
        try:
            with zipfile.ZipFile(yolo_zip_path, 'r') as zip_ref:
                zip_ref.extractall(temp_labels_dir)
        except Exception as e:
            print(f"  {batch:8} | Error extracting zip: {e}")
            continue
        
        # Load images
        image_files = [f for f in os.listdir(batch_path) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.JPG'))]
        images_loaded = 0
        batch_class_counts = Counter()
        
        for image_file in image_files:
            image_path = os.path.join(batch_path, image_file)
            label_file = image_file.replace('.JPG', '.txt').replace('.jpg', '.txt').replace('.jpeg', '.txt').replace('.png', '.txt')
            label_path = os.path.join(temp_labels_dir, label_file)
            
            # Try alternative naming
            if not os.path.exists(label_path):
                alt_label_file = "Copy of " + label_file
                alt_label_path = os.path.join(temp_labels_dir, alt_label_file)
                if os.path.exists(alt_label_path):
                    label_path = alt_label_path
                else:
                    continue
            
            image, labels = load_image_with_labels(image_path, label_path, class_mapping)
            
            if image and labels:
                class_counts = Counter(labels)
                primary_class = max(class_counts.items(), key=lambda x: x[1])[0]
                
                aerial_data.append({
                    'image_path': image_path,
                    'primary_class': primary_class,
                    'folder': 'aerial',
                    'batch': batch
                })
                images_loaded += 1
                batch_class_counts[primary_class] += 1
        
        # Store batch statistics
        batch_statistics[batch] = {
            'total': images_loaded,
            'unripe': batch_class_counts['Unripe'],
            'ripe': batch_class_counts['Ripe']
        }
        
        # Print batch details
        total_in_batch = batch_class_counts['Unripe'] + batch_class_counts['Ripe']
        if total_in_batch > 0:
            unripe_pct = (batch_class_counts['Unripe'] / total_in_batch) * 100
            ripe_pct = (batch_class_counts['Ripe'] / total_in_batch) * 100
            print(f"  {batch:8} | {images_loaded:3} images | Unripe: {batch_class_counts['Unripe']:3} ({unripe_pct:5.1f}%) | Ripe: {batch_class_counts['Ripe']:3} ({ripe_pct:5.1f}%)")
        else:
            print(f"  {batch:8} | {images_loaded:3} images | No Ripe/Unripe images")
    
    return aerial_data, batch_statistics

Below code loads both, the images from aerial and side folders and combines them into one dataset

In [ ]:
# Load all data
print("=== LOADING COMPLETE DATASET ===\n")
side_data, side_stats = load_side_images()
aerial_data, aerial_stats = load_aerial_images()

# Combine datasets
complete_dataset = side_data + aerial_data

As you can see, Side has 10 batches and Ariel has 5 batches.

For each side batches, we have some images (ripe and unripe)\
Side Batch is imbalanced (more unripe images than ripe images)

For each ariel batches, we have some images (ripe and unripe)\
Ariel Batch is imbalanced (more ripe images than Unripe images)

In [ ]:
# Final summary
print(f"\n" + "=" * 50)
print("=== FINAL DATASET SUMMARY ===")
print("=" * 50)

total_side = len(side_data)
total_aerial = len(aerial_data)
total_all = len(complete_dataset)

print(f"\nTotal images: {total_all}")
print(f"Side images: {total_side} ({total_side/total_all*100:.1f}%)")
print(f"Aerial images: {total_aerial} ({total_aerial/total_all*100:.1f}%)")

# Overall class distribution
class_counts = Counter([item['primary_class'] for item in complete_dataset])
print(f"\nOverall Class Distribution:")
for class_name, count in class_counts.items():
    percentage = (count / total_all) * 100
    print(f"  {class_name}: {count:4} images ({percentage:5.1f}%)")

# Print combined batch statistics
print(f"\n" + "=" * 50)
print("BATCH-WISE SUMMARY:")
print("=" * 50)
print(f"{'Batch':12} | {'Total':5} | {'Unripe':8} | {'Ripe':8}")
print("-" * 50)

all_batches = {**side_stats, **aerial_stats}
for batch_name in sorted(all_batches.keys()):
    stats = all_batches[batch_name]
    unripe_pct = (stats['unripe'] / stats['total']) * 100 if stats['total'] > 0 else 0
    ripe_pct = (stats['ripe'] / stats['total']) * 100 if stats['total'] > 0 else 0
    print(f"{batch_name:12} | {stats['total']:5} | {stats['unripe']:3} ({unripe_pct:4.1f}%) | {stats['ripe']:3} ({ripe_pct:4.1f}%)")

print(f"\nDataset ready for processing!")

As the final summary suggests, both the Ariel and Side images datasets are imbalanced. This could affect the performance of our models, so we may need to consider techniques such as resampling or using class weights during model training to address this issue.

## Machine Learning Models

### Step 2: Data Preprocessing and Feature Engineering for Machine Learning

First, we import all the necessary libraries required for data preprocessing and feature engineering. This includes libraries for image processing, data manipulation, and machine learning.

In [ ]:
import cv2
import numpy as np
from skimage import feature, transform
from sklearn.model_selection import train_test_split
import joblib

The below function extracts three important features from each image: Color Features (mean and std of each channel), Texture Features (LBP), and Shape features (HOG). These features are crucial for distinguishing between ripe and unripe coffee cherries based on their visual characteristics.

The extracted features are then combined into a single feature vector for each image, which will be used as input for the machine learning models. The function returns a list of feature vectors.

In [ ]:
def extract_features(image_paths, target_size=(128, 128)):
    """
    Extract features for classical ML models
    """
    features = []
    
    print(f"Extracting features for {len(image_paths)} images...")
    
    for i, image_path in enumerate(image_paths):
        if i % 100 == 0:  # Progress indicator
            print(f"  Processed {i}/{len(image_paths)} images...")
            
        try:
            # Load and preprocess image
            image = cv2.imread(image_path)
            if image is None:
                continue
                
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            
            # Resize to consistent size
            image = transform.resize(image, target_size, anti_aliasing=True)
            
            # Extract multiple features
            image_features = []
            
            # 1. Color features (mean and std of each channel)
            color_mean = np.mean(image, axis=(0, 1))
            color_std = np.std(image, axis=(0, 1))
            image_features.extend(color_mean)
            image_features.extend(color_std)
            
            # 2. Texture features (LBP)
            gray = cv2.cvtColor((image * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
            lbp = feature.local_binary_pattern(gray, 24, 3, method='uniform')
            lbp_hist, _ = np.histogram(lbp.ravel(), bins=26, range=(0, 25))
            lbp_hist = lbp_hist.astype(float)
            lbp_hist /= (lbp_hist.sum() + 1e-8)  # Normalize
            image_features.extend(lbp_hist)
            
            # 3. Shape features (HOG)
            hog_features = feature.hog(gray, pixels_per_cell=(16, 16), cells_per_block=(2, 2), 
                                      block_norm='L2-Hys', feature_vector=True)
            image_features.extend(hog_features)
            
            features.append(image_features)
            
        except Exception as e:
            print(f"Error processing {image_path}: {e}")
            continue
    
    print(f"Feature extraction complete! Processed {len(features)} images.")
    return np.array(features)

In [ ]:
# Prepare data for feature extraction
print("Preparing data for feature extraction...")
image_paths = [item['image_path'] for item in complete_dataset]
labels = [item['primary_class'] for item in complete_dataset]

# Convert labels to binary (0=Unripe, 1=Ripe)
label_mapping = {'Unripe': 0, 'Ripe': 1}
y = np.array([label_mapping[label] for label in labels])

print(f"Data prepared:")
print(f" - Total images: {len(image_paths)}")
print(f" - Unripe (0): {np.sum(y == 0)}")
print(f" - Ripe (1): {np.sum(y == 1)}")

### Step 3: Splitting the Dataset into Train/Validation/Test Split of 70/15/15

Next, we split the data into: Training, Validation, and Testing in the ratios 70:15:15

First, we make the training dataset to have 70% of all the data

Then, the rest of the data is split into 50-50 (15% each) for validation dataset and testing dataset

In [ ]:
# First split: 70% train, 30% temp (validation + test)
X_train_paths, X_temp_paths, y_train, y_temp = train_test_split(
    image_paths, y, test_size=0.3, random_state=42, stratify=y
)

# Second split: 50/50 of the 30% to get 15% validation and 15% test
X_val_paths, X_test_paths, y_val, y_test = train_test_split(
    X_temp_paths, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

Below code snippet summerizes the number of images in each split and also shows the number of ripe and unripe in each of the splits

In [ ]:
print(f"\n=== DATA SPLIT SUMMARY ===")
print(f"Training set: {len(X_train_paths)} images ({len(X_train_paths)/len(image_paths)*100:.1f}%)")
print(f"  - Unripe: {np.sum(y_train == 0)}, Ripe: {np.sum(y_train == 1)}")

print(f"Validation set: {len(X_val_paths)} images ({len(X_val_paths)/len(image_paths)*100:.1f}%)")
print(f"  - Unripe: {np.sum(y_val == 0)}, Ripe: {np.sum(y_val == 1)}")

print(f"Test set: {len(X_test_paths)} images ({len(X_test_paths)/len(image_paths)*100:.1f}%)")
print(f"  - Unripe: {np.sum(y_test == 0)}, Ripe: {np.sum(y_test == 1)}")

# Verify percentages
total = len(image_paths)
print(f"\nVerification:")
print(f"Train: {len(X_train_paths)/total*100:.1f}% (should be 70%)")
print(f"Val: {len(X_val_paths)/total*100:.1f}% (should be 15%)")
print(f"Test: {len(X_test_paths)/total*100:.1f}% (should be 15%)")

The summary suggests that we have successfully split the dataset into training, validation, and test sets with a ratio of 70:15:15. This split ensures that we have enough data for training the models while also having sufficient data for validating and testing their performance.

### Step 4: Feature Extraction for ML for all Splits

In the below code snippet, we call the extract_feature function (defined above) for each of the split

In [ ]:
print("\nStarting feature extraction for all splits...")

# Extract features for each split
print("1. Extracting training features...")
X_train_features = extract_features(X_train_paths)

print("\n2. Extracting validation features...")
X_val_features = extract_features(X_val_paths)

print("\n3. Extracting test features...")
X_test_features = extract_features(X_test_paths)

print(f"\n=== FEATURE SHAPES ===")
print(f"Training features: {X_train_features.shape}")
print(f"Validation features: {X_val_features.shape}")
print(f"Test features: {X_test_features.shape}")
print(f"Each image represented by {X_train_features.shape[1]} features")

Next, we check for NaN values and manage them.

Since we are expecting very few NaN values, we can remove (or drop) those columns that have NaN.

In [ ]:
# Check for any NaN values and handle them
print(f"\nData quality check:")
print(f"NaN in training: {np.isnan(X_train_features).sum()}")
print(f"NaN in validation: {np.isnan(X_val_features).sum()}")
print(f"NaN in test: {np.isnan(X_test_features).sum()}")

# Replace any NaN values with 0 (should be very few if any)
if np.isnan(X_train_features).sum() > 0:
    X_train_features = np.nan_to_num(X_train_features)
    X_val_features = np.nan_to_num(X_val_features)
    X_test_features = np.nan_to_num(X_test_features)
    print("Replaced NaN values with 0")

### Step 5: Balancing the Dataset

Since the dataset is imbalanced, we will apply techniques to balance it. We will use RandomUnderSampler to under sample the Majority class by removing the Noice using KNN.

This will help improve the performance of our machine learning models by providing them with a more balanced dataset to learn from.

We have to use RandomUnderSampler only on the training set to avoid data leakage. The validation and test sets should remain unchanged to provide an accurate evaluation of the model's performance.

The RandomUnderSampler is available in the imblearn library.

In [ ]:
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTEENN
from imblearn.under_sampling import RandomUnderSampler
from collections import Counter

This shows the current ratio of our classes

In [ ]:
print("=== HANDLING CLASS IMBALANCE ===")
print(f"Current class distribution:")
print(f"Training set - Unripe: {np.sum(y_train == 0)}, Ripe: {np.sum(y_train == 1)}")
print(f"Imbalance ratio: {np.sum(y_train == 0)/np.sum(y_train == 1):.1f}:1")

Now, we apply the RandomUnderSampler to our training features and labels to create balanced training data features and labels

In [ ]:
# Apply Random Under Sampler to balance the training data

print("\nApplying Random Under Sampler to balance the training data...")
rus = RandomUnderSampler(random_state=42)
X_train_balanced, y_train_balanced = rus.fit_resample(X_train_features, y_train)

print(f"After Random Under Sampler:")
print(f"Training set - Unripe: {np.sum(y_train_balanced == 0)}, Ripe: {np.sum(y_train_balanced == 1)}")
print(f"Total training samples: {len(X_train_balanced)}")
print(f"New balance ratio: {np.sum(y_train_balanced == 0)/np.sum(y_train_balanced == 1):.1f}:1")

# Let's also keep the original imbalanced data for comparison
X_train_imbalanced = X_train_features
y_train_imbalanced = y_train

print("\nTraining data balanced successfully with Random Under Sampler!")

### Step 6: Saving the Datasets

We can save the preprocessed and balanced datasets (training, validation, and test sets) to disk for future use. This will allow us to easily load the datasets when training and evaluating our machine learning models without having to repeat the preprocessing steps.

In [ ]:
# Create directory for processed data
import os
processed_dir = "processed_data"
os.makedirs(processed_dir, exist_ok=True)

# Save all the processed data
print(f"\nSaving processed data to '{processed_dir}'...")

joblib.dump({
    'X_train_balanced': X_train_balanced,
    'y_train_balanced': y_train_balanced,
    'X_train_imbalanced': X_train_imbalanced,
    'y_train_imbalanced': y_train_imbalanced,
    'X_val': X_val_features,
    'y_val': y_val,
    'X_test': X_test_features,
    'y_test': y_test,
    'feature_names': ['color_features', 'texture_features', 'shape_features']
}, os.path.join(processed_dir, 'ml_features.joblib'))

# Save the image paths for reference
joblib.dump({
    'train_paths': X_train_paths,
    'val_paths': X_val_paths,
    'test_paths': X_test_paths,
    'label_mapping': label_mapping
}, os.path.join(processed_dir, 'image_paths.joblib'))

print("Data saved successfully!")
print("Ready for machine learning model training!")

### Step 7: Training and Evaluating the Machine Learning Models

In this subsection, we will implement and evaluate two machine learning models for classifying coffee plantation images into ripe and unripe categories. The two models will be Random Forest and SVM.

First, we will import all the necassary libraries required for building and evaluating the machine learning models. This includes libraries for model implementation, evaluation metrics, and data manipulation.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

The following function will evaluate the model (both Random Forest and SVM) by first training the model on the training data, then making predictions on the validation and test sets. It will calculate and print various performance metrics such as accuracy, precision, recall, F1-score, and confusion matrix for both validation and test sets. This will help us understand how well the model is performing in classifying the coffee plantation images into ripe and unripe categories.

In [ ]:
def evaluate_model(model, X_train, y_train, X_val, y_val, X_test, y_test, model_name):
    """
    Train and evaluate a model with comprehensive metrics
    """
    print(f"\n{'='*50}")
    print(f"Training {model_name}...")
    
    # Train the model
    model.fit(X_train, y_train)
    
    # Predictions
    y_train_pred = model.predict(X_train)
    y_val_pred = model.predict(X_val)
    y_test_pred = model.predict(X_test)
    
    # Calculate metrics
    metrics = {}
    for split_name, y_true, y_pred in [("Training", y_train, y_train_pred), 
                                       ("Validation", y_val, y_val_pred), 
                                       ("Test", y_test, y_test_pred)]:
        metrics[split_name] = {
            'accuracy': accuracy_score(y_true, y_pred),
            'precision': precision_score(y_true, y_pred, average='binary', zero_division=0),
            'recall': recall_score(y_true, y_pred, average='binary', zero_division=0),
            'f1': f1_score(y_true, y_pred, average='binary', zero_division=0)
        }
    
    # Print results
    print(f"\n{model_name} Results:")
    print(f"{'Split':12} | {'Accuracy':8} | {'Precision':8} | {'Recall':8} | {'F1-Score':8}")
    print("-" * 60)
    for split_name, metric in metrics.items():
        print(f"{split_name:12} | {metric['accuracy']:8.4f} | {metric['precision']:8.4f} | "
              f"{metric['recall']:8.4f} | {metric['f1']:8.4f}")
    
    # Confusion Matrix for Test set
    cm = confusion_matrix(y_test, y_test_pred)
    plt.figure(figsize=(15, 4))
    
    plt.subplot(1, 3, 1)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Unripe', 'Ripe'], 
                yticklabels=['Unripe', 'Ripe'])
    plt.title(f'{model_name} - Test Set Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    
    # Feature Importance (for Random Forest)
    if hasattr(model, 'feature_importances_'):
        plt.subplot(1, 3, 2)
        feature_importance = model.feature_importances_
        top_features = np.argsort(feature_importance)[-10:]  # Top 10 features
        plt.barh(range(len(top_features)), feature_importance[top_features])
        plt.yticks(range(len(top_features)), [f'Feature {i}' for i in top_features])
        plt.title(f'{model_name} - Top 10 Feature Importance')
        plt.xlabel('Importance')
    
    # Class distribution in predictions
    plt.subplot(1, 3, 3)
    pred_counts = np.bincount(y_test_pred)
    true_counts = np.bincount(y_test)
    classes = ['Unripe', 'Ripe']
    
    x = np.arange(len(classes))
    width = 0.35
    
    plt.bar(x - width/2, true_counts, width, label='True', alpha=0.7)
    plt.bar(x + width/2, pred_counts, width, label='Predicted', alpha=0.7)
    plt.xlabel('Classes')
    plt.ylabel('Count')
    plt.title('True vs Predicted Distribution')
    plt.xticks(x, classes)
    plt.legend()
    
    plt.tight_layout()
    plt.show()
    
    # Detailed classification report
    print(f"\nDetailed Classification Report for Test Set:")
    print(classification_report(y_test, y_test_pred, 
                               target_names=['Unripe', 'Ripe'],
                               digits=4))
    
    return model, metrics, y_test_pred

Next, we initialize each models with class weights to handle imbalance

#### Step 7.1: Random Forest

In [ ]:
# Random Forest with class weights
rf_model = RandomForestClassifier(
    n_estimators=50,
    max_depth=8,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight='balanced',  # Handles imbalance
    random_state=42
)

#### Step 7.2: Support Vector Machine (SVM)

In [ ]:
# SVM with class weights and probability for better results
svm_model = SVC(
    kernel='rbf',
    C=1.0,
    class_weight='balanced',  # Handles imbalance
    probability=True,
    random_state=42
)

Next we train and evaluate both models

#### Step 7.3: Evaluating Random Forest

In [ ]:
# Train on BALANCED data (using RandomUnderSampler)
rf_trained, rf_metrics, rf_test_pred = evaluate_model(
    rf_model, X_train_balanced, y_train_balanced, X_val_features, y_val, X_test_features, y_test, 
    "Random Forest (Balanced)"
)

#### Step 7.4: Evaluating Support Vector Machine (SVM)

In [ ]:
# Train on BALANCED data (using RandomUnderSampler)
svm_trained, svm_metrics, svm_test_pred = evaluate_model(
    svm_model, X_train_balanced, y_train_balanced, X_val_features, y_val, X_test_features, y_test,
    "SVM (Balanced)"
)

#### Step 7.5: Comparison of Machine Learning Model Performance 

In [ ]:
models_comparison = {
    'Random Forest': rf_metrics,
    'SVM': svm_metrics
}

In [ ]:
# Compare model performance
print("\n" + "="*60)
print("MODEL PERFORMANCE COMPARISON")
print("="*60)

# Test set comparison
print("\nTest Set Performance Comparison:")
print(f"{'Model':15} | {'Accuracy':8} | {'Precision':8} | {'Recall':8} | {'F1-Score':8}")
print("-" * 60)
for model_name, metrics in models_comparison.items():
    test_metrics = metrics['Test']
    print(f"{model_name:15} | {test_metrics['accuracy']:8.4f} | {test_metrics['precision']:8.4f} | "
          f"{test_metrics['recall']:8.4f} | {test_metrics['f1']:8.4f}")

# Save the trained models
print(f"\nSaving trained models...")
joblib.dump(rf_trained, os.path.join(processed_dir, 'random_forest_model.joblib'))
joblib.dump(svm_trained, os.path.join(processed_dir, 'svm_model.joblib'))

print("Models saved successfully!")
print("Machine Learning implementation complete!")

## Deep Learning Model

In this subsection, we will implement one Deep learning model for classifying coffee plantation images into ripe and unripe categories. The model will be a Convolutional Neural Network (CNN).

Since, I am training on my local machine, which is M1 Macbook Air, I will keep the model simple with fewer layers and parameters to ensure it can be trained efficiently without running into memory or processing power issues.

In [ ]:
import tensorflow as tf
print(f"TensorFlow version: {tf.__version__}")

keras = tf.keras
layers = keras.layers
models = keras.models
ImageDataGenerator = keras.preprocessing.image.ImageDataGenerator
EarlyStopping = keras.callbacks.EarlyStopping
ReduceLROnPlateau = keras.callbacks.ReduceLROnPlateau

import matplotlib.pyplot as plt

print("GPU available:", tf.config.list_physical_devices('GPU'))
print("Metal available:", tf.config.list_physical_devices('METAL'))
print("All TensorFlow modules imported successfully")

### Step 8: Data Preprocessing for CNN

The below code is used to load and preprocess the images for training the CNN model. It involves resizing the images to a consistent size, normalizing the pixel values, and converting the labels into a format suitable for training.

We use Keras for the preprocessing of images.

In [ ]:
def load_and_preprocess_images(image_paths, labels, target_size=(128, 128)):
    """
    Load and preprocess images for CNN
    """
    images = []
    processed_labels = []
    
    print(f"\nLoading {len(image_paths)} images for CNN...")
    
    for i, (image_path, label) in enumerate(zip(image_paths, labels)):
        if i % 100 == 0:
            print(f"  Processed {i}/{len(image_paths)} images...")
            
        try:
            # Load image
            image = tf.keras.preprocessing.image.load_img(image_path, target_size=target_size)
            image_array = tf.keras.preprocessing.image.img_to_array(image)
            
            # Normalize to [0, 1]
            image_array = image_array / 255.0
            
            images.append(image_array)
            processed_labels.append(label)
            
        except Exception as e:
            print(f"Error loading {image_path}: {e}")
            continue
    
    return np.array(images), np.array(processed_labels)

Now, we use the already split and balanced datasets from Machine Learning section for training, validation, and testing of CNN model.

The only thing which changes in this section is data preprocessing, where we need to ensure that the images are in the correct format for input into the CNN model.

In [ ]:
# Prepare CNN data - USE THE BALANCED DATA from ML preprocessing
print("Loading BALANCED images for CNN training...")

# Get the indices that were selected by RandomUnderSampler
rus = RandomUnderSampler(random_state=42)
_, _ = rus.fit_resample(np.arange(len(X_train_paths)).reshape(-1, 1), y_train)
balanced_indices = rus.sample_indices_

# Use only the balanced subset for CNN training
balanced_train_paths = [X_train_paths[i] for i in balanced_indices]
balanced_train_labels = y_train_balanced  # This is already balanced

X_train_cnn, y_train_cnn = load_and_preprocess_images(balanced_train_paths, balanced_train_labels)
X_val_cnn, y_val_cnn = load_and_preprocess_images(X_val_paths, y_val)
X_test_cnn, y_test_cnn = load_and_preprocess_images(X_test_paths, y_test)

print(f"\nBALANCED CNN Data Shapes:")
print(f"Training: {X_train_cnn.shape}, Labels: {y_train_cnn.shape}")
print(f"Validation: {X_val_cnn.shape}, Labels: {y_val_cnn.shape}")
print(f"Test: {X_test_cnn.shape}, Labels: {y_test_cnn.shape}")

print(f"\nCNN Training Class Distribution:")
print(f"Unripe: {np.sum(y_train_cnn == 0)}, Ripe: {np.sum(y_train_cnn == 1)}")

### Step 9: Creating CNN Model  

The below function creates a simple Convolutional Neural Network (CNN) model using Keras. The model consists of 4 layers of convolutional and max-pooling layers, followed by a flattening layer and three dense layers. The final dense layer uses a sigmoid activation function for binary classification (ripe vs unripe). The model is compiled with the relu optimizer, binary cross-entropy loss function, and accuracy as the evaluation metric. This model will be trained on the preprocessed image data to classify coffee plantation images into ripe and unripe categories.

In [ ]:
def create_cnn_model(input_shape=(128, 128, 3)):
    """
    Create a CNN model for coffee classification
    """
    model = models.Sequential([
        # First Convolutional Block
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Second Convolutional Block
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Third Convolutional Block
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Fourth Convolutional Block
        layers.Conv2D(256, (3, 3), activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Flatten and Dense Layers
        layers.Flatten(),
        layers.Dense(512, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid')  # Binary classification
    ])
    
    return model

Next, we call the model we generated above

In [ ]:
# Create model
cnn_model = create_cnn_model()
print("CNN Model Architecture:")
cnn_model.summary()

# Compile model with class weights to handle imbalance
print("\nCompiling CNN model...")

# Calculate class weights for imbalance
from sklearn.utils.class_weight import compute_class_weight
class_weights = compute_class_weight(
    'balanced',
    classes=np.unique(y_train_cnn),
    y=y_train_cnn
)
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}

print(f"Class weights: {class_weight_dict}")

cnn_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', 'precision', 'recall']
)

# Callbacks
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=5,
    min_lr=1e-7,
    verbose=1
)

### Step 10: Training the CNN Model

In [ ]:
# Data augmentation to handle imbalance and improve generalization
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.2,
    shear_range=0.2,
    fill_mode='nearest'
)

In the following code snippet, we train the CNN model using the preprocessed training data. We use 50 epochs for training. The model is trained on the training dataset and validated on the validation dataset to monitor its performance during training.

In [ ]:
# Train the model with balanced approach
print("Starting CNN training...")

# Using sample_weight in a simpler way
sample_weight = compute_sample_weight(class_weight='balanced', y=y_train_cnn)

# Create a dataset with sample weights
train_dataset = tf.data.Dataset.from_tensor_slices((X_train_cnn, y_train_cnn, sample_weight))
train_dataset = train_dataset.batch(32).prefetch(tf.data.AUTOTUNE)

# For validation, no sample weights needed
val_dataset = tf.data.Dataset.from_tensor_slices((X_val_cnn, y_val_cnn))
val_dataset = val_dataset.batch(32).prefetch(tf.data.AUTOTUNE)

history = cnn_model.fit(
    train_dataset,
    epochs=50,
    validation_data=val_dataset,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

print("CNN training completed!")

### Step 11: Evaluating the CNN Model

In [ ]:
# Plot training history
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 3, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 3, 3)
plt.plot(history.history['precision'], label='Training Precision')
plt.plot(history.history['val_precision'], label='Validation Precision')
plt.plot(history.history['recall'], label='Training Recall')
plt.plot(history.history['val_recall'], label='Validation Recall')
plt.title('Precision and Recall')
plt.xlabel('Epoch')
plt.ylabel('Score')
plt.legend()

plt.tight_layout()
plt.show()

# Evaluate on all sets
print("Evaluating CNN model on all datasets...")

Below function evaluates the trained CNN model on the test dataset. It calculates and prints various performance metrics such as accuracy, precision, recall, F1-score, and confusion matrix. This will help us understand how well the CNN model is performing in classifying the coffee plantation images into ripe and unripe categories.

In [ ]:
def evaluate_cnn_model(model, X, y, dataset_name):
    predictions = model.predict(X)
    y_pred = (predictions > 0.5).astype(int).flatten()
    
    accuracy = accuracy_score(y, y_pred)
    precision = precision_score(y, y_pred, zero_division=0)
    recall = recall_score(y, y_pred, zero_division=0)
    f1 = f1_score(y, y_pred, zero_division=0)
    cm = confusion_matrix(y, y_pred)
    
    print(f"\n{dataset_name} Results:")
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    
    # Plot confusion matrix
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Unripe', 'Ripe'], 
                yticklabels=['Unripe', 'Ripe'])
    plt.title(f'CNN - {dataset_name} Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()
    
    return accuracy, precision, recall, f1, y_pred

We call the evaluate_cnn_model function to evaluate the performance of the trained CNN model on the test dataset. This will provide us with insights into how well the model generalizes to unseen data and its effectiveness in classifying coffee plantation images into ripe and unripe categories.

In [ ]:
# Evaluate on all datasets
train_acc, train_prec, train_rec, train_f1, _ = evaluate_cnn_model(cnn_model, X_train_cnn, y_train_cnn, "Training")
val_acc, val_prec, val_rec, val_f1, _ = evaluate_cnn_model(cnn_model, X_val_cnn, y_val_cnn, "Validation")
test_acc, test_prec, test_rec, test_f1, test_pred = evaluate_cnn_model(cnn_model, X_test_cnn, y_test_cnn, "Test")

# Save CNN model
print("\nSaving CNN model...")
cnn_model.save(os.path.join(processed_dir, 'cnn_model.h5'))
print("CNN model saved successfully!")

### Step 12: Final Comparison of All Models

In [ ]:
# Collect all test results
all_models_results = {
    'Random Forest': rf_metrics['Test'],
    'SVM': svm_metrics['Test'],
    'CNN': {
        'accuracy': test_acc,
        'precision': test_prec,
        'recall': test_rec,
        'f1': test_f1
    }
}

Finally, we can summerize and compare the performance of all the models: Random Forest, SVM, and CNN. This comparison will help us determine which model performs best for classifying coffee plantation images into ripe and unripe categories based on the evaluation metrics obtained from the test dataset.

In [ ]:
print("\n" + "="*70)
print("FINAL MODEL COMPARISON - RANDOM FOREST vs SVM vs CNN")
print("="*70)

print(f"\n{'Model':15} | {'Accuracy':8} | {'Precision':8} | {'Recall':8} | {'F1-Score':8}")
print("-" * 70)
for model_name, metrics in all_models_results.items():
    print(f"{model_name:15} | {metrics['accuracy']:8.4f} | {metrics['precision']:8.4f} | "
          f"{metrics['recall']:8.4f} | {metrics['f1']:8.4f}")

print("\nALL MODELS TRAINED AND EVALUATED SUCCESSFULLY!")